# Inferir datos Churn Client
Generalmente las inferencias en produción son realizadas con el modelo `@Champion` o `@Production`.

## Consumir/usar el modelo para inferencias batch
Ahora que el modelo esta registrado en UC puede realizar inferencias y guardarlas en una tabla para construir dashboards.

Utilizamos caracteristicas de ingenieria, automaticamente cargando PySpark UDF.
Para realizar predicciones batch utilice el método `score_batch`.

## Crear inferencias en notebook
Si planea ejecutar inferencias batch en el ambiente por defecto Serverless (`env_manager="local"`), en otro caso puede correr en un ambiente virtual (`env_manager="virtual_env", "uv", "condo"`) sacado de los artefactos requeridos por el modelo.

## 1. Configuración

from mlflow.store.artifact.models_artifact_repo import ModelsArtifactRepository

requirements_path = ModelsArtifactRepository(model_uri).download_artifacts(artifact_path="requirements.txt")

%pip install --quiet -r $requirements_path
%restart_pytho

In [0]:
import sys
python_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
print(f"Versión de Python actual: {python_version}")

Versión de Python actual: 3.12.3


In [0]:
import os
import mlflow
from mlflow.tracking.client import MlflowClient

catalog = "medallion_dev"
schema = "gold"
model_name = "customer_churn_classifier"
model_full_name = f"{catalog}.{schema}.{model_name}"
model_version = 1
model_alias = "Production" # "Challenger", "Champion" 

client = MlflowClient()
model_details = client.get_model_version_by_alias(model_full_name, model_alias)

if model_details:
    model_version = int(model_details.version)
    model_uri = f"models:/{model_full_name}@{model_alias}"
else:
    print(f"El modelo {model_full_name} no tiene alias {model_alias}.")
    model_details = client.get_model_version(model_full_name, model_version)
    model_uri = f"models:/{model_full_name}/{model_version}"

if not model_details:
    raise Exception("No encontrado version del modelo registrado en UC.")

requirements_path = mlflow.artifacts.download_artifacts(artifact_uri=f"runs:/{model_details.run_id}/model/requirements.txt") 

if os.path.exists(requirements_path):
    print(f"Se encontró archivo requirements.txt en {requirements_path}")
    with open(requirements_path, 'r') as f:
        requirements_content = f.read()
    print(requirements_content)
else:
    print(f"No se encontró archivo requirements.txt en {requirements_path}")
    requirements_path = mlflow.artifacts.download_artifacts(artifact_uri=f"{model_uri}/model/requirements.txt") 
    if os.path.exists(requirements_path):
        print(f"Opcion 2. Se encontró archivo requirements.txt en {requirements_path}")
    else:
        print(f"Opcion 2. No se encontró archivo requirements.txt en {requirements_path}")

Se encontró archivo requirements.txt en /local_disk0/user_tmp_data/spark-8bfb4eba-9467-497c-8c85-e3/tmpckobjusy/model/requirements.txt
mlflow==3.12.0
cloudpickle==3.0.0
scipy==1.17.1
scikit-learn==1.8.0
lightgbm==4.6.0
xgboost==3.2.0


In [0]:
# Instalar paquetes desde el requirements.txt del modelo
%pip install --quiet -r $requirements_path

# Instalar dependencia adicional de Databricks
%pip install --quiet databricks-feature-engineering

dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
catalog = "medallion_dev"
schema = "gold"
model_name = "customer_churn_classifier"
model_full_name = f"{catalog}.{schema}.{model_name}"
model_version = 1
feature_table_name = f"{catalog}.{schema}.churn_user_features"
target_col = "churn"

endpoint_name = f"{model_name}_endpoint"
model_alias = "Production" # "Challenger", "Champion" 
model_uri = f"models:/{model_full_name}@{model_alias}"
table_offline_inferences = "Churn_Offline_Inference"
rng_seed=42

model_uri = f"models:/{model_full_name}/{model_version}"
#model_uri = f"models:/{model_full_name}@{model_alias}"

#spark.sql(f"USE CATALOG {catalog}")
#spark.sql(f"USE `{catalog}`.`{schema}`")

## 2. Dataframe

In [0]:
supported_cols = ["gender", "last_transaction", "canal", "event_count", "days_since_creation", "country", "session_count",
    "order_count", "days_last_event", "days_since_last_activity", "total_amount", "total_item", "age_group", "platform"]

df = spark.table(feature_table_name).sample(fraction=0.1, seed=rng_seed).limit(100)
result_type=df.schema[target_col].dataType
df = df.select(*supported_cols, target_col)
pdf = df.toPandas()

#display(df.limit(5)) 

## 3. Inferencia con Pandas

In [0]:
import mlflow
import pandas as pd

try:
    # Cargar modelo
    model = mlflow.pyfunc.load_model(model_uri)

    # Predecir
    y_pred = model.predict(pdf)
    pdf['prediction'] = y_pred

    display(pdf.head(5))

    # Crear DataFrame con resultados
    test_results = pdf.copy()
    test_results['prediction'] = y_pred

#    y_pred_proba = model.predict_proba(pdf)[:, 1]
#    test_results['prediction_proba'] = y_pred_proba

    print(f"\nPredicciones completadas: {len(test_results)} registros")
    display(test_results.head(5))

    # Métricas
    from sklearn.metrics import classification_report
    y_test = pdf[target_col]
    
    print(f"\n{classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'])}")
except Exception as e:
    print(f"Error al realizar prediciones con Pandas. Error: {e}")

2026/05/09 03:33:27 WARNING mlflow.models.utils: Found extra inputs in the model input that are not defined in the model signature: `['prediction', 'churn']`. These inputs will be ignored.
/local_disk0/.ephemeral_nfs/envs/pythonEnv-8bfb4eba-9467-497c-8c85-e33e5861aae0/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


gender,last_transaction,canal,event_count,days_since_creation,country,session_count,order_count,days_last_event,days_since_last_activity,total_amount,total_item,age_group,platform,churn,prediction
0,2023-06-05T06:23:07.000Z,PHONE,4,1185,SPAIN,3,4,1039,1039,223,9,4,ios,1,1
1,2023-06-08T02:08:17.000Z,PHONE,1,1377,USA,0,1,1041,1043,96,3,5,other,1,1
1,2023-06-08T21:50:54.000Z,WEBAPP,2,1489,SPAIN,1,2,1038,1041,103,5,5,ios,0,0
1,2023-06-07T22:53:49.000Z,PHONE,2,1068,USA,2,2,1039,1039,66,3,3,ios,1,1
1,2023-06-03T11:22:04.000Z,WEBAPP,2,1564,USA,1,1,1037,1040,38,1,8,android,1,1



Predicciones completadas: 100 registros


gender,last_transaction,canal,event_count,days_since_creation,country,session_count,order_count,days_last_event,days_since_last_activity,total_amount,total_item,age_group,platform,churn,prediction
0,2023-06-05T06:23:07.000Z,PHONE,4,1185,SPAIN,3,4,1039,1039,223,9,4,ios,1,1
1,2023-06-08T02:08:17.000Z,PHONE,1,1377,USA,0,1,1041,1043,96,3,5,other,1,1
1,2023-06-08T21:50:54.000Z,WEBAPP,2,1489,SPAIN,1,2,1038,1041,103,5,5,ios,0,0
1,2023-06-07T22:53:49.000Z,PHONE,2,1068,USA,2,2,1039,1039,66,3,3,ios,1,1
1,2023-06-03T11:22:04.000Z,WEBAPP,2,1564,USA,1,1,1037,1040,38,1,8,android,1,1



              precision    recall  f1-score   support

    No Churn       0.76      0.53      0.63        30
       Churn       0.82      0.93      0.87        70

    accuracy                           0.81       100
   macro avg       0.79      0.73      0.75       100
weighted avg       0.80      0.81      0.80       100



## 4. Inferencia con Spark UDF

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
import mlflow

fe = FeatureEngineeringClient()

def spark_udf_predict(df, model_alias: str, result_type: str, useEnvironment=False, env_manager="local"):
    """Aplica modelo a un DataFrame Spark 
    Args:
        df (DataFrame): DataFrame Spark
        model_alias (str): Alias del modelo
        result_type (str): Tipo de resultado
        useEnvironment (bool): Usar entornos
        env_manager (str): ['local', 'conda', 'virtualenv', 'uv']
          'virtualenv': Muy demorado porque tiene que instala: Python y muchos paquetes
          'conda': Usar entornos conda. Databricks Serverless no soporta env_manager = 'conda'
          'uv': Usar entornos universales. Usa Pyton, pero tiene que instalar muchos paquetes
          'local': Usar entornos locales. Recomendado para Databricks Serverless No requiere instalar paquetes
    Returns:
        DataFrame: DataFrame Spark con columna de predicciones
    """

    if useEnvironment:
      predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{model_full_name}@{model_alias}", result_type=result_type, env_manager=env_manager)
    else:
      predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri=f"models:/{model_full_name}@{model_alias}", result_type=result_type)

    return df.withColumn('prediction', predict_udf(*predict_udf.metadata.get_input_schema().input_names()))
    #return df.withColumn("prediction", predict_udf(*[col for col in df.columns if col != target_col]))

try:
    predict_df = spark_udf_predict(df, model_alias, result_type, False)
    #predict_df = spark_udf_predict(df, model_alias, result_type, True, "local")

    display(predict_df.limit(5))

    print(f"\nDistribución de la variable objetivo '{target_col}':")
    temp_df = predict_df.groupBy(target_col).count().withColumnRenamed("count", "Contar")
    total_count = predict_df.count()
    temp_df = temp_df.withColumn("Porcentaje", (temp_df["Contar"] / total_count) * 100)
    display(temp_df.orderBy(target_col))
except Exception as e:
    print(f"Error al realizar predicciones con Spark. Mensaje: {e}")

2026/05/09 03:43:30 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2026/05/09 03:43:31 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


gender,last_transaction,canal,event_count,days_since_creation,country,session_count,order_count,days_last_event,days_since_last_activity,total_amount,total_item,age_group,platform,churn,prediction
0,2023-06-05T06:23:07.000Z,PHONE,4,1185,SPAIN,3,4,1039,1039,223,9,4,ios,1,1
1,2023-06-08T02:08:17.000Z,PHONE,1,1377,USA,0,1,1041,1043,96,3,5,other,1,1
1,2023-06-08T21:50:54.000Z,WEBAPP,2,1489,SPAIN,1,2,1038,1041,103,5,5,ios,0,0
1,2023-06-07T22:53:49.000Z,PHONE,2,1068,USA,2,2,1039,1039,66,3,3,ios,1,1
1,2023-06-03T11:22:04.000Z,WEBAPP,2,1564,USA,1,1,1037,1040,38,1,8,android,1,1



Distribución de la variable objetivo 'churn':


churn,Contar,Porcentaje
0,30,30.0
1,70,70.0


## 5. Inferencia batch

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient(model_registry_uri="databricks-uc")

def batch_predict(df, model_alias: str, result_type: str, useEnvironment=False, env_manager="local"):
    """Aplica modelo a un DataFrame Spark 
    Args:
        df (DataFrame): DataFrame Spark
        model_alias (str): Alias del modelo
        result_type (str): Tipo de resultado
        useEnvironment (bool): Usar entornos
        env_manager (str): ['local', 'conda', 'virtualenv', 'uv']
          'virtualenv': Muy demorado porque tiene que instala: Python y muchos paquetes
          'conda': Usar entornos conda. Databricks Serverless no soporta env_manager = 'conda'
          'uv': Usar entornos universales. Usa Pyton, pero tiene que instalar muchos paquetes
          'local': Usar entornos locales. Recomendado para Databricks Serverless No requiere instalar paquetes
    Returns:
        DataFrame: DataFrame Spark con columna de predicciones
    """
    if useEnvironment:
        predict_df = fe.score_batch(model_uri=f"models:/{model_full_name}@{model_alias}", df=df, result_type=result_type, env_manager=env_manager)
    else:
        predict_df = fe.score_batch(model_uri=f"models:/{model_full_name}@{model_alias}", df=df, result_type=result_type)
    return predict_df

try:
    predict_batch_df = batch_predict(df, model_alias, result_type, False)
    #predict_batch_df = batch_predict(df, model_alias, result_type, True, "local")

    display(predict_batch_df.limit(5))
except Exception as e:
    print(f"Error al realizar predicciones con Spark. Mensaje: {e}")

2026/05/09 03:41:23 INFO mlflow.store.artifact.artifact_repo: Failed to complete request, possibly due to credential expiration (Error: 404 Client Error: Not Found for url: http://storage-proxy.databricks.com/api/2.0/fs/files/Models/medallion_dev/gold/customer_churn_classifier/1/data/feature_store/feature_spec.yaml?v=1&workspaceId=7474651374019400&wkt=cp&userId=77345568936916&X-Databricks-TTL=1800000&X-Databricks-Consumer-Network-Zone=databricks-data-plane&sn=ohio-prod&X-Databricks-Key-Version=01f105ef-46f7-150e-96d4-2cd66d9f5329&X-Databricks-Issued=20260509T034123Z&X-Databricks-Signature=AewRINZdQa3FZ_OugrgR6brHik3WOh-5M7YBLaAjjGrm_LAjVw. Response text: {
  "error_code" : "NOT_FOUND",
  "message" : "The file being accessed is not found.",
  "details" : [ {
    "@type" : "type.googleapis.com/google.rpc.ErrorInfo",
    "reason" : "FILES_API_FILE_NOT_FOUND",
    "domain" : "filesystem.databricks.com"
  } ]
}). Refreshing credentials and trying again...


Error al realizar predicciones con Spark. Mensaje: The following failures occurred while downloading one or more artifacts from /Models/medallion_dev/gold/customer_churn_classifier/1:
##### File data/feature_store/feature_spec.yaml #####
404 Client Error: Not Found for url: http://storage-proxy.databricks.com/api/2.0/fs/files/Models/medallion_dev/gold/customer_churn_classifier/1/data/feature_store/feature_spec.yaml?v=1&workspaceId=7474651374019400&wkt=cp&userId=77345568936916&X-Databricks-TTL=1800000&X-Databricks-Consumer-Network-Zone=databricks-data-plane&sn=ohio-prod&X-Databricks-Key-Version=01f105ef-46f7-150e-96d4-2cd66d9f5329&X-Databricks-Issued=20260509T034123Z&X-Databricks-Signature=AewRINZdQa3FZ_OugrgR6brHik3WOh-5M7YBLaAjjGrm_LAjVw. Response text: {
  "error_code" : "NOT_FOUND",
  "message" : "The file being accessed is not found.",
  "details" : [ {
    "@type" : "type.googleapis.com/google.rpc.ErrorInfo",
    "reason" : "FILES_API_FILE_NOT_FOUND",
    "domain" : "filesystem.da

## 6. Guardar las predicciones para monitoreo
- Para monitorear el modelo y sus predicciones con el tiempo, guardamos las predicciones con información del modelo. 
- Esta tabla puede ser monitoreada para desviacion de caracteristicas y desviación de predicciones. 
- Las desviaciones son una señal que el modelo tiene que ser reentrenado.

In [0]:
from mlflow import MlflowClient
from datetime import datetime, timedelta
from pyspark.sql import functions as F

client = MlflowClient()

model = client.get_registered_model(name=model_full_name)
model_version = client.get_model_version_by_alias(name=model_full_name, alias=model_alias).version

offline_inference_df = predict_df \
                              .withColumn("model_version", F.lit(model_version)) \
                              .withColumn("inference_timestamp", F.lit(datetime.now()))

offline_inference_df.write.mode("append") \
                    .option("overwriteSchema", True) \
                    .saveAsTable(table_offline_inferences)

display(offline_inference_df.limit(5))

gender,last_transaction,canal,event_count,days_since_creation,country,session_count,order_count,days_last_event,days_since_last_activity,total_amount,total_item,age_group,platform,churn,prediction,model_version,inference_timestamp
0,2023-06-05T06:23:07.000Z,PHONE,4,1185,SPAIN,3,4,1039,1039,223,9,4,ios,1,1,1,2026-05-09T03:44:23.930Z
1,2023-06-08T02:08:17.000Z,PHONE,1,1377,USA,0,1,1041,1043,96,3,5,other,1,1,1,2026-05-09T03:44:23.930Z
1,2023-06-08T21:50:54.000Z,WEBAPP,2,1489,SPAIN,1,2,1038,1041,103,5,5,ios,0,0,1,2026-05-09T03:44:23.930Z
1,2023-06-07T22:53:49.000Z,PHONE,2,1068,USA,2,2,1039,1039,66,3,3,ios,1,1,1,2026-05-09T03:44:23.930Z
1,2023-06-03T11:22:04.000Z,WEBAPP,2,1564,USA,1,1,1037,1040,38,1,8,android,1,1,1,2026-05-09T03:44:23.930Z


## 7. Inferencia con payloads via peticiones REST
Puede probar el endpoint en la UI: Serving > Abra el endpoint > Use

- Browser: Adicione JSON:
</br>
{
    "dataframe_records": [
          {"user_id": "1", "age_group": 2, "canal": "WEBAPP", "country": "SPAIN", "gender": 0, "platform": "ios", "event_count": 4, "session_count": 4, "order_count": 3, "total_amount": 152, "total_item": 9, "last_transaction": "2023-06-08 21:50:54", "days_since_creation": 1044, "days_since_last_activity": 1044, "days_last_event": 1036},
          {"user_id": "2", "age_group": 8, "canal": "MOBILE", "country": "USA", "gender": 1, "platform": "ios", "event_count": 6, "session_count": 3, "order_count": 4, "total_amount": 155, "total_item": 2, "last_transaction": "2023-06-06 20:19:26", "days_since_creation": 1477, "days_since_last_activity": 1040, "days_last_event": 1044},
          {"user_id": "3", "age_group": 5, "canal": "PHONE", "country": "FR", "gender": 0, "platform": "ios", "event_count": 2, "session_count": 1, "order_count": 2, "total_amount": 100, "total_item": 1, "last_transaction": "2023-06-06 20:19:26", "days_since_creation": 1589, "days_since_last_activity": 1141, "days_last_event": 1144}
    ]
}

- Curl:
curl \
  -u token:$DATABRICKS_TOKEN \
  -X POST \
  -H "Content-Type: application/json" \
  -d@data.json \
  https://dbc-c4a7b866-5204.cloud.databricks.com/serving-endpoints/customer_churn_classifier_endpoint/invocations 

In [0]:
import os
token = os.environ.get("...")
display(token)

In [0]:
import os
import requests
import numpy as np
import pandas as pd
import json

# Ejemplo de conjunto de datos para predicción
data = [
    {"user_id": "1", "age_group": 2, "canal": "WEBAPP", "country": "SPAIN", "gender": 0, "platform": "ios", "event_count": 4, "session_count": 4, "order_count": 3, "total_amount": 152, "total_item": 9, "last_transaction": "2023-06-08 21:50:54", "days_since_creation": 1044, "days_since_last_activity": 1044, "days_last_event": 1036},
    {"user_id": "2", "age_group": 8, "canal": "MOBILE", "country": "USA", "gender": 1, "platform": "ios", "event_count": 6, "session_count": 3, "order_count": 4, "total_amount": 155, "total_item": 2, "last_transaction": "2023-06-06 20:19:26", "days_since_creation": 1477, "days_since_last_activity": 1040, "days_last_event": 1044},
    {"user_id": "3", "age_group": 5, "canal": "PHONE", "country": "FR", "gender": 0, "platform": "ios", "event_count": 2, "session_count": 1, "order_count": 2, "total_amount": 100, "total_item": 1, "last_transaction": "2023-06-06 20:19:26", "days_since_creation": 1589, "days_since_last_activity": 1141, "days_last_event": 1144}
]
dataset = pd.DataFrame(data)

def create_tf_serving_json(data):
    return {'inputs': {name: data[name].tolist() for name in data.keys()} if isinstance(data, dict) else data.tolist()}

def score_model(dataset):
    # token = "..."
    # Debe suministrar el token como variable de entorno o quemarlo dentro del codigo (no recomendado)
    token = os.environ.get("DATABRICKS_TOKEN")
    if not token:
        raise Exception("No se encontró el token en la variable de entorno DATABRICKS_TOKEN.")

    url = 'https://dbc-c4a7b866-5204.cloud.databricks.com/serving-endpoints/customer_churn_classifier_endpoint/invocations'
    headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
    ds_dict = {'dataframe_split': dataset.to_dict(orient='split')} if isinstance(dataset, pd.DataFrame) else create_tf_serving_json(dataset)
    data_json = json.dumps(ds_dict, allow_nan=True)
    response = requests.request(method='POST', headers=headers, url=url, data=data_json)
    if response.status_code != 200:
        raise Exception(f'Request failed with status {response.status_code}, {response.text}')
    return response.json()

try:
    response_df = score_model(dataset)
    print("Respuesta del servicio Endpoint:")
    print(response_df)
except Exception as e:
    print(f"Error al realizar predicciones. Mensaje: {e}")

Error al realizar predicciones. Mensaje: No se encontró el token en la variable de entorno DATABRICKS_TOKEN.


In [0]:
%sql
SELECT ai_query(
    endpoint => 'customer_churn_classifier_endpoint',
    request => named_struct("total_item", 9, "last_transaction", "2023-06-08 21:50:54", "age_group", 2, "days_since_creation", 1044, "days_since_last_activity", 1044, "country", "SPAIN", "session_count", 4, "order_count", 3, "days_last_event", 1036, "event_count", 4, "platform", "ios", "total_amount", 152, "canal", "WEBAPP", "gender", 0),
    returnType => 'DOUBLE'
) AS prediction

prediction
1.0


In [0]:
import mlflow.deployments
client = mlflow.deployments.get_deploy_client("databricks")

try:
    response_df = client.predict(
        endpoint=endpoint_name,
        inputs={
            "dataframe_records": [
        {"user_id": "1", "age_group": 2, "canal": "WEBAPP", "country": "SPAIN", "gender": 0, "platform": "ios", "event_count": 4, "session_count": 4, "order_count": 3, "total_amount": 152, "total_item": 9, "last_transaction": "2023-06-08 21:50:54", "days_since_creation": 1044, "days_since_last_activity": 1044, "days_last_event": 1036},
        {"user_id": "2", "age_group": 8, "canal": "MOBILE", "country": "USA", "gender": 1, "platform": "ios", "event_count": 6, "session_count": 3, "order_count": 4, "total_amount": 155, "total_item": 2, "last_transaction": "2023-06-06 20:19:26", "days_since_creation": 1477, "days_since_last_activity": 1040, "days_last_event": 1044},
        {"user_id": "3", "age_group": 5, "canal": "PHONE", "country": "FR", "gender": 0, "platform": "ios", "event_count": 2, "session_count": 1, "order_count": 2, "total_amount": 100, "total_item": 1, "last_transaction": "2023-06-06 20:19:26", "days_since_creation": 1589, "days_since_last_activity": 1141, "days_last_event": 1144}
            ]
        },
    )

    print("Respuesta del servicio Endpoint:")
    print(response_df)
except Exception as e:
    print(f"Error al realizar predicciones con Endpoint. Mensaje: {e}")    

Respuesta del servicio Endpoint:
{'predictions': [1, 1, 0]}


In [0]:
from mlflow.store.artifact.models_artifact_repo import ModelsArtifactRepository
from mlflow.models import Model
import time
from databricks.sdk import WorkspaceClient
import pandas as pd

try:
  data = [
        {"user_id": "1", "age_group": 2, "canal": "WEBAPP", "country": "SPAIN", "gender": 0, "platform": "ios", "event_count": 4, "session_count": 4, "order_count": 3, "total_amount": 152, "total_item": 9, "last_transaction": "2023-06-08 21:50:54.000", "days_since_creation": 1044, "days_since_last_activity": 1044, "days_last_event": 1036},
        {"user_id": "2", "age_group": 8, "canal": "MOBILE", "country": "USA", "gender": 1, "platform": "ios", "event_count": 6, "session_count": 3, "order_count": 4, "total_amount": 155, "total_item": 2, "last_transaction": "2023-06-06 20:19:26", "days_since_creation": 1477, "days_since_last_activity": 1040, "days_last_event": 1044},
        {"user_id": "3", "age_group": 5, "canal": "PHONE", "country": "FR", "gender": 0, "platform": "ios", "event_count": 2, "session_count": 1, "order_count": 2, "total_amount": 100, "total_item": 1, "last_transaction": "2023-06-06 20:19:26", "days_since_creation": 1589, "days_since_last_activity": 1141, "days_last_event": 1144}
  ]

  print("Respuesta del servicio endpoint:")

  # Usar WorkspaceClient para autenticación automática en notebooks (no requiere crear token)
  # Uso de JSON
  w = WorkspaceClient()
  response = w.serving_endpoints.query(name=endpoint_name, dataframe_records=data)

  #time.sleep(10)
  print(response.predictions)

  # Uso de Dataframe
  dataset = pd.DataFrame(data)
  dataframe_records = dataset.to_dict(orient='records')
  response = w.serving_endpoints.query(name=endpoint_name, dataframe_records=dataframe_records)
  print(response)
except Exception as e:
    print(f"Error al realizar predicciones con Endpoint. Mensaje: {e}")

Respuesta del servicio endpoint:

[1, 1, 0]
QueryEndpointResponse(choices=[], created=None, data=[], id=None, model=None, object=None, outputs=None, predictions=[1, 1, 0], served_model_name='customer_churn_classifier-1', usage=None)


## 8. Crear inferencias Web con Postman

### Crear token
Icono usuario > Settings > Developer > “Acces token” > Manage > “Generate new token”: 
- Nombre
- Scope: “Others APIs”
- Lifetime: Numero de días
- API scope(s): “Model-serving” (para restringir a endpoint que servicios de modelos)

### URL
URLDatabricks usuarios: Inicio de url hasta ...databricks.com/
Endpoint: endpoint_name
URLEndpoint: .../serving-endpoints/customer_churn_classifier_endpoint/invocations

### Ejeuctar desde Postmant
- Metodo POST: URLEndpoint
- Authorizations: Bearer Token: Escribir token
- Body: JSON (Columnas FS sin incluir la columna a predecir)
- Ejemplo de Body:

{
    "dataframe_records": [
          {"user_id": "1", "age_group": 2, "canal": "WEBAPP", "country": "SPAIN", "gender": 0, "platform": "ios", "event_count": 4, "session_count": 4, "order_count": 3, "total_amount": 152, "total_item": 9, "last_transaction": "2023-06-08 21:50:54", "days_since_creation": 1044, "days_since_last_activity": 1044, "days_last_event": 1036},
          {"user_id": "2", "age_group": 8, "canal": "MOBILE", "country": "USA", "gender": 1, "platform": "ios", "event_count": 6, "session_count": 3, "order_count": 4, "total_amount": 155, "total_item": 2, "last_transaction": "2023-06-06 20:19:26", "days_since_creation": 1477, "days_since_last_activity": 1040, "days_last_event": 1044},
          {"user_id": "3", "age_group": 5, "canal": "PHONE", "country": "FR", "gender": 0, "platform": "ios", "event_count": 2, "session_count": 1, "order_count": 2, "total_amount": 100, "total_item": 1, "last_transaction": "2023-06-06 20:19:26", "days_since_creation": 1589, "days_since_last_activity": 1141, "days_last_event": 1144}
    ]
}

Postman retorna JSON: {"predictions": [1, 1, 0]}